# Multi-Architecture Comparison — TCN / LSTM / Mamba VAE on SMD
Trains all three models on the selected machines and shows live progress.
Results are saved to `results/comparison.json` after each model completes (crash-safe).

In [ ]:
import os, json, time, warnings
import numpy as np
import torch
import torch.nn as nn
from torch.optim import Adam
from torch.optim.lr_scheduler import ReduceLROnPlateau
from scipy.ndimage import uniform_filter1d
from sklearn.metrics import f1_score, roc_auc_score
from concurrent.futures import ThreadPoolExecutor, as_completed
import pandas as pd
from IPython.display import display, clear_output
from tqdm.notebook import tqdm

warnings.filterwarnings('ignore', category=UserWarning)

from preprocess      import load_smd_machine, build_dataloader_from_array
from vae_model       import VAE,      ELBOLoss, compute_reconstruction_errors as tcn_mse
from lstm_model      import LSTMVAE,  compute_reconstruction_errors as lstm_mse
from mamba_model     import MambaVAE, compute_reconstruction_errors as mamba_mse
from evaluation      import pot_threshold, point_adjusted_f1
from results_logger  import RunLogger

## Section 1 — Configuration
Edit `MACHINES` and `EPOCHS` before running. Everything else is shared hyperparameters.

In [ ]:
# ── Which machines / architectures to run ──────────────────────────────────
MACHINES = ['machine-1-1']          # swap for ALL_MACHINES (defined below) for full sweep
ARCHS    = ['TCN', 'LSTM', 'Mamba'] # subset e.g. ['TCN'] to skip slow models
DATA_DIR = './data/ServerMachineDataset'
OUT_DIR  = './results'
EPOCHS   = 100

# ── Shared hyperparameters ─────────────────────────────────────────────────
HP = dict(
    window_size   = 64,
    latent_dim    = 32,
    batch_size    = 256,
    warmup_epochs = 25,
    max_beta      = 1.0,
    lr            = 1e-3,
    smooth_kernel = 5,
    stride_train  = 1,
    stride_calib  = 64,
)

# ── Per-architecture backbone hyperparameters ──────────────────────────────
ARCH_HP = dict(
    TCN   = dict(tcn_hidden=64, n_heads=4),
    LSTM  = dict(lstm_hidden=128, num_layers=2, n_heads=4),
    Mamba = dict(d_model=64, d_state=16, n_layers=4, expand=2, n_heads=4),
)

print(f"Epochs: {EPOCHS} | Machines: {MACHINES} | Archs: {ARCHS}")

In [ ]:
# ── OmniAnomaly published PA-F1 baselines (Su et al., KDD 2019) ─────────────
OMNI_BASELINES = {
    'machine-1-1':  {'pa_f1': 0.8383}, 'machine-1-2':  {'pa_f1': 0.8533},
    'machine-1-3':  {'pa_f1': 0.9238}, 'machine-1-4':  {'pa_f1': 0.9469},
    'machine-1-5':  {'pa_f1': 0.9031}, 'machine-1-6':  {'pa_f1': 0.8763},
    'machine-1-7':  {'pa_f1': 0.8824}, 'machine-1-8':  {'pa_f1': 0.8156},
    'machine-2-1':  {'pa_f1': 0.9286}, 'machine-2-2':  {'pa_f1': 0.9011},
    'machine-2-3':  {'pa_f1': 0.9195}, 'machine-2-4':  {'pa_f1': 0.9355},
    'machine-2-5':  {'pa_f1': 0.9124}, 'machine-2-6':  {'pa_f1': 0.9167},
    'machine-2-7':  {'pa_f1': 0.9407}, 'machine-2-8':  {'pa_f1': 0.9524},
    'machine-2-9':  {'pa_f1': 0.9063}, 'machine-3-1':  {'pa_f1': 0.9143},
    'machine-3-2':  {'pa_f1': 0.8979}, 'machine-3-3':  {'pa_f1': 0.9302},
    'machine-3-4':  {'pa_f1': 0.9215}, 'machine-3-5':  {'pa_f1': 0.9286},
    'machine-3-6':  {'pa_f1': 0.9118}, 'machine-3-7':  {'pa_f1': 0.9483},
    'machine-3-8':  {'pa_f1': 0.9231}, 'machine-3-9':  {'pa_f1': 0.9412},
    'machine-3-10': {'pa_f1': 0.9167}, 'machine-3-11': {'pa_f1': 0.9302},
}
ALL_MACHINES = list(OMNI_BASELINES.keys())

## Section 2 — Helper Functions

In [ ]:
def build_model(arch: str, in_channels: int) -> nn.Module:
    if arch == 'TCN':
        return VAE(in_channels=in_channels, latent_dim=HP['latent_dim'],
                   window_size=HP['window_size'], **ARCH_HP['TCN'])
    elif arch == 'LSTM':
        return LSTMVAE(in_channels=in_channels, latent_dim=HP['latent_dim'],
                       window_size=HP['window_size'], **ARCH_HP['LSTM'])
    elif arch == 'Mamba':
        return MambaVAE(in_channels=in_channels, latent_dim=HP['latent_dim'],
                        window_size=HP['window_size'], **ARCH_HP['Mamba'])
    else:
        raise ValueError(f'Unknown architecture: {arch}')


def score_model(arch: str, model, dataloader, device) -> torch.Tensor:
    fns = {'TCN': tcn_mse, 'LSTM': lstm_mse, 'Mamba': mamba_mse}
    return fns[arch](model, dataloader, device)


def train_model(model, dataloader, device, epochs, lr, warmup_epochs, max_beta, label=''):
    """Train with a tqdm epoch bar showing live loss values."""
    model.to(device)
    optimizer = Adam(model.parameters(), lr=lr)
    scheduler = ReduceLROnPlateau(optimizer, patience=5, factor=0.5)
    criterion = ELBOLoss(warmup_epochs=warmup_epochs, max_beta=max_beta)

    history = {'total': [], 'recon': [], 'kl': [], 'beta': []}
    pbar = tqdm(range(epochs), desc=label, leave=True, dynamic_ncols=True)

    for epoch in pbar:
        model.train()
        rt, rr, rk, n = 0.0, 0.0, 0.0, 0
        for batch in dataloader:
            batch = batch.to(device)
            x_mu, x_log_var, z_mu, z_log_var = model(batch)
            loss, recon, kl = criterion(batch, x_mu, x_log_var, z_mu, z_log_var,
                                        current_epoch=epoch)
            optimizer.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            rt += loss.item(); rr += recon.item(); rk += kl.item(); n += 1

        avg_t, avg_r, avg_k = rt/n, rr/n, rk/n
        beta = criterion.get_beta(epoch)
        history['total'].append(avg_t)
        history['recon'].append(avg_r)
        history['kl'].append(avg_k)
        history['beta'].append(beta)
        scheduler.step(avg_t)

        pbar.set_postfix(loss=f'{avg_t:.4f}', recon=f'{avg_r:.4f}',
                         kl=f'{avg_k:.5f}', beta=f'{beta:.2f}')

    return history


def evaluate(arch, model, train_data, test_data, test_labels,
             train_mean, train_std, device):
    W, BS, SK = HP['window_size'], HP['batch_size'], HP['smooth_kernel']

    calib_loader = build_dataloader_from_array(
        train_data, window_size=W, stride=W, batch_size=BS,
        shuffle=False, train_mean=train_mean, train_std=train_std)
    train_scores = score_model(arch, model, calib_loader, device).numpy()
    threshold = float(np.nanpercentile(train_scores, 99))

    test_loader = build_dataloader_from_array(
        test_data, window_size=W, stride=1, batch_size=BS,
        shuffle=False, train_mean=train_mean, train_std=train_std)
    test_scores_raw = score_model(arch, model, test_loader, device).numpy()
    test_scores = uniform_filter1d(test_scores_raw.astype(float), size=SK)

    n_windows = len(test_scores)
    aligned_labels = test_labels[W-1 : W-1+n_windows]

    nan_frac = float(np.isnan(test_scores).mean())
    if nan_frac > 0:
        print(f'  WARNING: {nan_frac*100:.1f}% NaN in test scores')

    preds = (test_scores > threshold).astype(int)
    valid = ~np.isnan(test_scores)

    base = dict(threshold=threshold, nan_frac=nan_frac,
                _train_scores=train_scores, _test_scores_raw=test_scores_raw,
                _test_scores=test_scores, _preds=preds, _labels=aligned_labels)

    if valid.sum() == 0 or aligned_labels[valid].sum() == 0:
        return dict(f1=float('nan'), pa_f1=float('nan'), roc_auc=float('nan'), **base)

    raw_f1 = f1_score(aligned_labels[valid], preds[valid], zero_division=0)
    pa_res = point_adjusted_f1(preds[valid], aligned_labels[valid])
    try:
        roc = roc_auc_score(aligned_labels[valid], test_scores[valid])
    except Exception:
        roc = float('nan')

    return dict(f1=raw_f1, pa_f1=pa_res['f1'], roc_auc=roc,
                pa_precision=float(pa_res.get('precision', 0)),
                pa_recall=float(pa_res.get('recall', 0)), **base)


print('Helper functions defined.')

## Section 3 — Parallel Data Loading
All machine data is loaded concurrently (I/O-bound — no GPU involved).

In [ ]:
def _load_one(machine):
    train_data, test_data, test_labels = load_smd_machine(DATA_DIR, machine)
    train_mean = train_data.mean(axis=0).astype(np.float32)
    train_std  = np.maximum(train_data.std(axis=0), 1e-3).astype(np.float32)
    return machine, (train_data, test_data, test_labels, train_mean, train_std)

machine_data = {}
failed = []

print(f'Loading {len(MACHINES)} machine(s) in parallel...')
with ThreadPoolExecutor(max_workers=min(8, len(MACHINES))) as ex:
    futures = {ex.submit(_load_one, m): m for m in MACHINES}
    for fut in tqdm(as_completed(futures), total=len(futures), desc='Loading data'):
        m = futures[fut]
        try:
            machine, data = fut.result()
            machine_data[machine] = data
        except FileNotFoundError as e:
            print(f'  SKIP {m}: {e}')
            failed.append(m)

print(f'Loaded: {len(machine_data)}  |  Skipped: {len(failed)}')
if machine_data:
    sample = next(iter(machine_data.values()))
    print(f'Train shape: {sample[0].shape}  Test shape: {sample[1].shape}  '
          f'Labels shape: {sample[2].shape}')

## Section 4 — Training Loop
Models train sequentially (single GPU). A live results table updates after each model finishes.

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if device.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')

os.makedirs(OUT_DIR, exist_ok=True)
results_path = os.path.join(OUT_DIR, 'comparison.json')

# Load existing results if resuming
if os.path.exists(results_path):
    with open(results_path) as f:
        all_results = json.load(f)
    completed = sum(1 for m in all_results if not m.startswith('_')
                    for a in [all_results[m]] if all(k+'-VAE' in a for k in ARCHS))
    print(f'Resuming from {results_path} — {completed} machines already complete')
else:
    all_results = {}
    print('Starting fresh run.')

In [ ]:
# Live results table — call after each model to refresh display
results_handle = display(pd.DataFrame(columns=['Waiting for results...']), display_id=True)

def _fmt(val):
    """Format a metric value, treating NaN as missing."""
    if val is None:
        return '---'
    try:
        v = float(val)
        if np.isnan(v):
            return '---'
        return f'{v:.4f}'
    except (TypeError, ValueError):
        return '---'

def refresh_table():
    rows = []
    for m in MACHINES:
        if m not in all_results:
            continue
        row = {'Machine': m}
        for arch in ARCHS:
            key = arch + '-VAE'
            r = all_results[m].get(key, {})
            row[f'{arch} PA-F1'] = _fmt(r.get('pa_f1'))
            row[f'{arch} F1']    = _fmt(r.get('f1'))
            row[f'{arch} AUC']   = _fmt(r.get('roc_auc'))
        row['OmniAnom PA-F1'] = f"{OMNI_BASELINES.get(m, {}).get('pa_f1', 0):.4f}"
        rows.append(row)
    if rows:
        df = pd.DataFrame(rows).set_index('Machine')
        results_handle.update(df)

refresh_table()

In [ ]:
for machine in MACHINES:
    if machine not in machine_data:
        print(f'[SKIP] {machine} — data not loaded')
        continue

    machine_results = all_results.get(machine, {})

    # Skip whole machine if all archs done
    if all((a + '-VAE') in machine_results for a in ARCHS):
        print(f'[SKIP] {machine} — all architectures already complete')
        continue

    print(f'\n{"="*60}\nMachine: {machine}\n{"="*60}')

    train_data, test_data, test_labels, train_mean, train_std = machine_data[machine]
    in_channels = train_data.shape[1]

    train_loader = build_dataloader_from_array(
        train_data,
        window_size = HP['window_size'],
        stride      = HP['stride_train'],
        batch_size  = HP['batch_size'],
        shuffle     = True,
        train_mean  = train_mean,
        train_std   = train_std,
    )

    for arch in ARCHS:
        key = arch + '-VAE'
        if key in machine_results:
            print(f'  [{arch}-VAE] SKIP — already done')
            continue

        print(f'\n  [{arch}-VAE] Building model...')
        model = build_model(arch, in_channels)
        n_params = sum(p.numel() for p in model.parameters())
        print(f'  [{arch}-VAE] Parameters: {n_params:,}')

        # ── TRAIN ──
        t0 = time.time()
        history = train_model(
            model, train_loader, device,
            epochs        = EPOCHS,
            lr            = HP['lr'],
            warmup_epochs = HP['warmup_epochs'],
            max_beta      = HP['max_beta'],
            label         = f'{machine} [{arch}-VAE]',
        )
        train_time = time.time() - t0
        print(f'  [{arch}-VAE] Training done in {train_time:.0f}s '
              f'— final loss: {history["total"][-1]:.4f}')

        # ── EVALUATE ──
        print(f'  [{arch}-VAE] Evaluating...')
        eval_result = evaluate(
            arch, model, train_data, test_data, test_labels,
            train_mean, train_std, device,
        )

        # Separate raw arrays from JSON-safe metrics
        raw_arrays = {k: eval_result.pop(k) for k in list(eval_result) if k.startswith('_')}
        metrics = eval_result
        metrics['train_time_s'] = round(train_time, 1)

        print(f'  [{arch}-VAE]  PA-F1={metrics["pa_f1"]:.4f}  '
              f'F1={metrics["f1"]:.4f}  '
              f'ROC-AUC={metrics["roc_auc"]:.4f}  '
              f't={train_time:.0f}s')

        # ── SAVE CHECKPOINT ──
        ckpt_path = os.path.join(OUT_DIR, f'{machine}_{arch.lower()}.pt')
        torch.save({'model_state_dict': model.state_dict(),
                    'metrics': metrics, 'machine': machine, 'arch': arch}, ckpt_path)

        # ── SAVE STRUCTURED RESULTS ──
        logger = RunLogger(machine, key, output_dir=OUT_DIR)
        logger.log_hyperparameters(**HP, **ARCH_HP.get(arch, {}))
        logger.log_training(history, train_time_s=train_time)
        logger.log_train_scores(raw_arrays['_train_scores'], metrics['threshold'],
                                threshold_pct99=metrics['threshold'])
        logger.log_test_results(
            raw_arrays['_test_scores_raw'], raw_arrays['_test_scores'],
            raw_arrays['_preds'], raw_arrays['_labels'],
            metrics['f1'], metrics['pa_f1'], metrics.get('roc_auc'),
        )
        logger.save()

        machine_results[key] = metrics

    all_results[machine] = machine_results

    # Crash-safe incremental write
    with open(results_path, 'w') as f:
        json.dump(all_results, f, indent=2)
    print(f'  Saved -> {results_path}')

    refresh_table()

print('\nAll done!')

## Section 5 — Summary Table

In [ ]:
rows = []
for machine in ALL_MACHINES:
    if machine not in all_results:
        continue
    row = {'Machine': machine}
    for arch in ['TCN', 'LSTM', 'Mamba']:
        key = arch + '-VAE'
        r = all_results[machine].get(key, {})
        row[f'{arch} PA-F1'] = r.get('pa_f1', float('nan'))
        row[f'{arch} F1']    = r.get('f1',    float('nan'))
        row[f'{arch} AUC']   = r.get('roc_auc', float('nan'))
    row['OmniAnom'] = OMNI_BASELINES.get(machine, {}).get('pa_f1', float('nan'))
    rows.append(row)

if rows:
    summary_df = pd.DataFrame(rows).set_index('Machine')

    # Append mean row
    mean_row = summary_df.mean(numeric_only=True).rename('MEAN')
    summary_df = pd.concat([summary_df, mean_row.to_frame().T])

    # Only highlight columns that have at least one non-NaN value
    pa_f1_cols = [c for c in summary_df.columns
                  if 'PA-F1' in c and summary_df[c].notna().any()]

    styler = summary_df.style.format('{:.4f}', na_rep='---')
    if pa_f1_cols:
        styler = (styler
                  .highlight_max(axis=0, subset=pa_f1_cols,
                                 props='font-weight: bold; color: green')
                  .highlight_min(axis=0, subset=pa_f1_cols,
                                 props='color: red'))
    display(styler)
else:
    print('No results to display yet.')

# Write baselines into JSON for paper table generation
all_results['_omni_baselines'] = OMNI_BASELINES
with open(results_path, 'w') as f:
    json.dump(all_results, f, indent=2)
print(f'Final results saved -> {results_path}')